In [5]:
!pip uninstall -y lightfm
!pip install lightfm-next

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.5 MB/s eta 0:00:00


In [1]:
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k

print("LightFM imported successfully!")

LightFM imported successfully!


In [4]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q ml-100k.zip

print("Dataset downloaded successfully")

replace ml-100k/allbut.pl? [y]es, [n]o, [A]ll, [N]one, [r]ename: Dataset downloaded successfully


In [9]:
import os
import joblib
import numpy as np
import pandas as pd

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score


RANDOM_STATE = 42
DATASET_DIR = "/content/ml-100k"
OUTPUT_DIR = "/content/lightfm_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# LOAD RATINGS
# ============================================================

ratings_df = pd.read_csv(
    os.path.join(DATASET_DIR, "u.data"),
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1"
)


# ============================================================
# LOAD MOVIES
# ============================================================

movie_columns = [
    "movie_id",
    "title",
    "release_date",
    "video_release_date",
    "imdb_url",
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

movies_df = pd.read_csv(
    os.path.join(DATASET_DIR, "u.item"),
    sep="|",
    names=movie_columns,
    encoding="latin-1"
)


# ============================================================
# LOAD USERS
# ============================================================

users_df = pd.read_csv(
    os.path.join(DATASET_DIR, "u.user"),
    sep="|",
    names=[
        "user_id",
        "age",
        "gender",
        "occupation",
        "zip_code"
    ],
    encoding="latin-1"
)


# ============================================================
# CLEAN DATA
# ============================================================

ratings_df = ratings_df.drop_duplicates().copy()

movies_df = movies_df.drop_duplicates(
    subset=["movie_id"]
).copy()

users_df = users_df.drop_duplicates(
    subset=["user_id"]
).copy()

movies_df["title"] = movies_df["title"].fillna(
    "Unknown Movie"
)

positive_ratings_df = ratings_df[
    ratings_df["rating"] >= 4
].copy()

print("Ratings:", ratings_df.shape)
print("Movies:", movies_df.shape)
print("Users:", users_df.shape)
print("Positive interactions:", len(positive_ratings_df))


# ============================================================
# CREATE LIGHTFM DATASET
# ============================================================

lightfm_dataset = Dataset()

lightfm_dataset.fit(
    users=users_df["user_id"].unique(),
    items=movies_df["movie_id"].unique()
)

interaction_tuples = list(
    positive_ratings_df[
        ["user_id", "movie_id"]
    ].itertuples(
        index=False,
        name=None
    )
)

interactions, _ = lightfm_dataset.build_interactions(
    interaction_tuples
)

train_interactions, test_interactions = (
    random_train_test_split(
        interactions,
        test_percentage=0.20,
        random_state=np.random.RandomState(
            RANDOM_STATE
        )
    )
)

print("Train interactions:", train_interactions.nnz)
print("Test interactions:", test_interactions.nnz)


# ============================================================
# TRAIN LIGHTFM MODEL
# ============================================================

model = LightFM(
    no_components=30,
    loss="warp",
    learning_rate=0.05,
    random_state=RANDOM_STATE
)

model.fit(
    train_interactions,
    epochs=15,
    num_threads=1,
    verbose=True
)

print("Model trained successfully")


# ============================================================
# EVALUATE MODEL
# ============================================================

train_precision = precision_at_k(
    model,
    train_interactions,
    k=10,
    num_threads=1
).mean()

test_precision = precision_at_k(
    model,
    test_interactions,
    train_interactions=train_interactions,
    k=10,
    num_threads=1
).mean()

test_recall = recall_at_k(
    model,
    test_interactions,
    train_interactions=train_interactions,
    k=10,
    num_threads=1
).mean()

test_auc = auc_score(
    model,
    test_interactions,
    train_interactions=train_interactions,
    num_threads=1
).mean()

print("\nEvaluation Results")
print("-" * 40)
print("Train Precision@10:", round(float(train_precision), 4))
print("Test Precision@10 :", round(float(test_precision), 4))
print("Test Recall@10    :", round(float(test_recall), 4))
print("Test AUC          :", round(float(test_auc), 4))


# ============================================================
# GET ID MAPPINGS
# ============================================================

user_id_map, _, item_id_map, _ = (
    lightfm_dataset.mapping()
)

reverse_item_id_map = {
    internal_id: raw_id
    for raw_id, internal_id
    in item_id_map.items()
}


# ============================================================
# CONVERT TRAIN MATRIX FOR ROW ACCESS
# ============================================================

train_interactions_csr = train_interactions.tocsr()


# ============================================================
# PRECOMPUTE TOP-20 RECOMMENDATIONS FOR EVERY USER
# ============================================================

recommendation_rows = []

number_of_items = len(item_id_map)
all_internal_item_ids = np.arange(number_of_items)

for raw_user_id, internal_user_id in user_id_map.items():

    scores = model.predict(
        internal_user_id,
        all_internal_item_ids,
        num_threads=1
    )

    watched_items = set(
        train_interactions_csr[
            internal_user_id
        ].indices
    )

    ranked_items = np.argsort(-scores)

    recommended_internal_ids = [
        item_id
        for item_id in ranked_items
        if item_id not in watched_items
    ][:20]

    for rank, internal_item_id in enumerate(
        recommended_internal_ids,
        start=1
    ):

        recommendation_rows.append(
            {
                "user_id": raw_user_id,
                "movie_id": reverse_item_id_map[
                    internal_item_id
                ],
                "rank": rank,
                "recommendation_score": float(
                    scores[internal_item_id]
                )
            }
        )

recommendations_df = pd.DataFrame(
    recommendation_rows
)

recommendations_df = recommendations_df.merge(
    movies_df[
        [
            "movie_id",
            "title",
            "release_date",
            "imdb_url"
        ]
    ],
    on="movie_id",
    how="left"
)

print(
    "Precomputed recommendations shape:",
    recommendations_df.shape
)

recommendations_df.head(10)



# ============================================================
# POPULAR MOVIES
# ============================================================

popular_movies_df = (
    ratings_df.groupby("movie_id")
    .agg(
        rating_count=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .reset_index()
    .merge(
        movies_df[
            [
                "movie_id",
                "title",
                "release_date",
                "imdb_url"
            ]
        ],
        on="movie_id",
        how="left"
    )
    .sort_values(
        ["rating_count", "average_rating"],
        ascending=False
    )
)


# ============================================================
# SAVE ARTIFACTS
# ============================================================

joblib.dump(
    model,
    os.path.join(
        OUTPUT_DIR,
        "lightfm_model.pkl"
    )
)

joblib.dump(
    train_interactions,
    os.path.join(
        OUTPUT_DIR,
        "train_interactions.pkl"
    )
)

joblib.dump(
    user_id_map,
    os.path.join(
        OUTPUT_DIR,
        "user_id_map.pkl"
    )
)

joblib.dump(
    item_id_map,
    os.path.join(
        OUTPUT_DIR,
        "item_id_map.pkl"
    )
)

joblib.dump(
    reverse_item_id_map,
    os.path.join(
        OUTPUT_DIR,
        "reverse_item_id_map.pkl"
    )
)

model_metrics = {
    "train_precision_at_10": float(
        train_precision
    ),
    "test_precision_at_10": float(
        test_precision
    ),
    "test_recall_at_10": float(
        test_recall
    ),
    "test_auc": float(test_auc),
    "number_of_users": int(
        interactions.shape[0]
    ),
    "number_of_movies": int(
        interactions.shape[1]
    ),
    "positive_interactions": int(
        interactions.nnz
    )
}

joblib.dump(
    model_metrics,
    os.path.join(
        OUTPUT_DIR,
        "model_metrics.pkl"
    )
)

recommendations_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "user_recommendations.csv"
    ),
    index=False
)

popular_movies_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "popular_movies.csv"
    ),
    index=False
)

movies_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "movies.csv"
    ),
    index=False
)

users_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "users.csv"
    ),
    index=False
)

ratings_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ratings.csv"
    ),
    index=False
)

print("\nAll files saved successfully!")
print("Output folder:", OUTPUT_DIR)

Ratings: (100000, 4)
Movies: (1682, 24)
Users: (943, 5)
Positive interactions: 55375
Train interactions: 44300
Test interactions: 11075


Epoch: 100%|██████████| 15/15 [00:00<00:00, 24.03it/s]


Model trained successfully

Evaluation Results
----------------------------------------
Train Precision@10: 0.5568
Test Precision@10 : 0.2102
Test Recall@10    : 0.2379
Test AUC          : 0.9377
Precomputed recommendations shape: (18860, 7)

All files saved successfully!
Output folder: /content/lightfm_outputs


In [12]:
improved_model = LightFM(
    no_components=50,
    loss="warp",
    learning_rate=0.03,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42
)

improved_model.fit(
    train_interactions,
    epochs=40,
    num_threads=1,
    verbose=True
)

Epoch: 100%|██████████| 40/40 [00:02<00:00, 16.96it/s]


In [13]:
improved_train_precision = precision_at_k(
    improved_model,
    train_interactions,
    k=10,
    num_threads=1
).mean()

improved_test_precision = precision_at_k(
    improved_model,
    test_interactions,
    train_interactions=train_interactions,
    k=10,
    num_threads=1
).mean()

improved_test_recall = recall_at_k(
    improved_model,
    test_interactions,
    train_interactions=train_interactions,
    k=10,
    num_threads=1
).mean()

improved_test_auc = auc_score(
    improved_model,
    test_interactions,
    train_interactions=train_interactions,
    num_threads=1
).mean()

print("Improved LightFM Results")
print("-" * 35)
print("Train Precision@10:", round(float(improved_train_precision), 4))
print("Test Precision@10 :", round(float(improved_test_precision), 4))
print("Test Recall@10    :", round(float(improved_test_recall), 4))
print("Test AUC          :", round(float(improved_test_auc), 4))

Improved LightFM Results
-----------------------------------
Train Precision@10: 0.5942
Test Precision@10 : 0.2177
Test Recall@10    : 0.2462
Test AUC          : 0.9399


In [14]:
comparison = pd.DataFrame({
    "Metric": [
        "Train Precision@10",
        "Test Precision@10",
        "Test Recall@10",
        "Test AUC"
    ],
    "Original Model": [
        train_precision,
        test_precision,
        test_recall,
        test_auc
    ],
    "Improved Model": [
        improved_train_precision,
        improved_test_precision,
        improved_test_recall,
        improved_test_auc
    ]
})

comparison

,Metric,Original Model,Improved Model
0,Train Precision@10,0.556794,0.594161
1,Test Precision@10,0.210206,0.217698
2,Test Recall@10,0.237884,0.246173
3,Test AUC,0.937693,0.939927


In [6]:
# Convert sparse matrices to CSR format for row-wise indexing
train_interactions_csr = train_interactions.tocsr()
test_interactions_csr = test_interactions.tocsr()

print("Train matrix format:", type(train_interactions_csr))
print("Test matrix format:", type(test_interactions_csr))

Train matrix format: <class 'scipy.sparse._csr.csr_matrix'>
Test matrix format: <class 'scipy.sparse._csr.csr_matrix'>


In [15]:
import os
import joblib

OUTPUT_DIR = "/content/lightfm_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save final improved model
joblib.dump(improved_model, os.path.join(OUTPUT_DIR, "lightfm_model.pkl"))

# Save interaction matrix
joblib.dump(train_interactions, os.path.join(OUTPUT_DIR, "train_interactions.pkl"))

# Save mappings
joblib.dump(user_id_map, os.path.join(OUTPUT_DIR, "user_id_map.pkl"))
joblib.dump(item_id_map, os.path.join(OUTPUT_DIR, "item_id_map.pkl"))
joblib.dump(reverse_item_id_map, os.path.join(OUTPUT_DIR, "reverse_item_id_map.pkl"))

# Save metrics
metrics = {
    "Train Precision@10": float(improved_train_precision),
    "Test Precision@10": float(improved_test_precision),
    "Test Recall@10": float(improved_test_recall),
    "Test AUC": float(improved_test_auc),
    "Users": 943,
    "Movies": 1682,
    "Ratings": 100000,
    "Positive Interactions": 55375
}

joblib.dump(metrics, os.path.join(OUTPUT_DIR, "model_metrics.pkl"))

# Save datasets
movies_df.to_csv(os.path.join(OUTPUT_DIR, "movies.csv"), index=False)
ratings_df.to_csv(os.path.join(OUTPUT_DIR, "ratings.csv"), index=False)
users_df.to_csv(os.path.join(OUTPUT_DIR, "users.csv"), index=False)
recommendations_df.to_csv(os.path.join(OUTPUT_DIR, "user_recommendations.csv"), index=False)
popular_movies_df.to_csv(os.path.join(OUTPUT_DIR, "popular_movies.csv"), index=False)

print("✅ Final model saved successfully!")

✅ Final model saved successfully!


In [16]:
!zip -r /content/lightfm_outputs.zip /content/lightfm_outputs > /dev/null

from google.colab import files
files.download("/content/lightfm_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>